# Exercise Part 1: Giving Your Agent a Memory

In this exercise, you will test the limits of a "stateless" LLM hitting its context limit, and then implement two strategies to stay within the budget: a sliding window and summarization.

## Setup

In [ ]:
# ! pip install transformers torch accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 200
MAX_CONTEXT_TOKENS = 500   # deliberately small
SYSTEM_PROMPT = "You are a helpful, concise travel assistant."

In [ ]:

# ═════════════════════════════════════════════════════════════════════════════
#  HELPERS — do not modify
# ═════════════════════════════════════════════════════════════════════════════

def load_model(model_id=MODEL_ID):
    print(f"Loading model: {model_id}\n")
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    tok = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=dtype,  # device_map="auto", low_cpu_mem_usage=True TODO: set device if available
    )
    mdl.eval()
    print(f"Model loaded on: {next(mdl.parameters()).device}\n")
    return mdl, tok


def generate(model, tokenizer, messages, max_new_tokens=MAX_NEW_TOKENS):
    """Run the model on a list of messages and return the reply text."""
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, temperature=None, top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()


def count_tokens(tokenizer, messages):
    """How many tokens this message list occupies once template is added."""
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    return len(tokenizer.encode(text))


def run_stateless(model, tokenizer, user_turns):
    """A chatbot with NO memory: each turn sees only the current user message."""
    print("=" * 64)
    print("STATELESS CHATBOT (no memory)")
    print("=" * 64)
    for turn in user_turns:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": turn},      # ← only THIS turn, nothing else
        ]
        reply = generate(model, tokenizer, messages)
        print(f"\n👨‍🦲 User: {turn}")
        print(f"🤖 Bot : {reply}")
    print()



Here, we will use scripted conversations (fixed lists of user messages) instead of live typing. This way everyone sees the same behavior and can compare results. You can switch to interactive mode at the end if you like.

In [ ]:

# Short conversation: the last turn requires remembering the first turns.
CONVERSATION_SHORT = [
    "Hi! My name is Alex and I'm planning a trip to Japan in March.",
    "I'm especially interested in temples and really good ramen.",
    "What's my name, and what was I asking about?",   # <- needs memory to answer
]

# Longer conversation: watch context grow
CONVERSATION_LONG = [
    "Hi! My name is Alex and I'm planning a trip to Japan in March.",
    "I'm especially interested in temples and really good ramen.",
    "My partner is travelling with me and they are vegetarian.",
    "We'd like to visit Kyoto and Osaka, maybe Tokyo too.",
    "We enjoy hiking and would love a day trip into nature.",
    "Our budget is moderate — nice but not luxury.",
    "We prefer trains over flights within the country.",
    "Remind me: what is my name, and is my partner vegetarian?",   # <- needs early turns
]

# ─────────────────────────────────────────────────────────────────────────────
# for cross-conversation task
# ─────────────────────────────────────────────────────────────────────────────
CONVERSATION_1 = [
    "Hi, I'm Alex. I'm planning a trip to Japan in March with my vegetarian partner.",
    "We love temples, ramen (veggie options matter!), and a bit of hiking.",
]
CONVERSATION_2 = [
    "What are 3 restaurants you'd recommend for my trip?",   # only makes sense if model remembers
]


## Background: Messages

A chat model sees a list of messages, each with a role and content:

```
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi, my name is Alex."},
    {"role": "assistant", "content": "Hello Alex! How can I help?"},
    {"role": "user", "content": "What is my name?"},   # needs the earlier turn to answer
]
```

The model only knows what's **in this list**. If you don't include the earlier turns, the model has no way to know your name, since there is no hidden memory.

Every message costs tokens, and the model has a maximum context length. When the conversation grows past that limit, something has to give.

## Task 0: Watch the model forget (2 mins)

Use the provided code to load the model and tun it as is. Run it with `run_stateless` on a short conversation where the user introduces themselves and later asks the model to recall a detail.

Can the model answer the recall question?

In [ ]:
# YOUR CODE HERE

model, tokenizer = load_model()

run_stateless(model, tokenizer, CONVERSATION_SHORT)

## Task 1: Short-Term Memory (8 mins)

Implement `run_with_memory`. Instead of sending only the current mesage, keep a growing list of all messages so the model sees the full conversation each turn.

In [ ]:
def run_with_memory(model, tokenizer, user_turns):
    """
    A chatbot WITH short-term memory.
    Returns the final messages list.
    """
    print("=" * 64)
    print("CHATBOT WITH SHORT-TERM MEMORY")
    print("=" * 64)

    # TODO: 
    #   1. Initialise `messages` with the system prompt.
    #   2. For each user turn:
    #        - append user message
    #        - reply = generate(model, tokenizer, messages)
    #        - append assistant message
    #        - print the turn, the reply, and count_tokens(tokenizer, messages)
    #   3. Return messages.

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    # YOUR CODE HERE
    return messages


Then test your code by running it on the same conversation. 

Does the model now recall the earlier information?

In [ ]:
# run_with_memory(model, tokenizer, CONVERSATION_SHORT)

## Task 2: Watch context grow, then slide the window (10 mins)

Real conversations do not fit the context forever. 
First, we will measure the growth.

1. **Measure**: The loop already prints `count_tokens(messages)` after each turn. Run `run_with_memory` on the longer conversation (`CONVERSATION_LONG`) and watch the token count increase.
2. **Slide the Window**: Implement `trim_with_sliding_window`. While the message list
is over `max_tokens`, **drop the oldest user/assistant pair** (but always keep the system
message). Plug it into `run_with_memory_managed`.

NOTE: We set a deliberately small `MAX_CONTEXT_TOKENS` (e.g. 500) so the limit triggers quickly.

In [ ]:
def trim_with_sliding_window(tokenizer, messages, max_tokens=MAX_CONTEXT_TOKENS):
    """
    While the messages exceed max_tokens and there are non-system messages left to drop, drop the OLDEST user/assistant pair.
    Always keep the system message (messages[0]).
    Returns the trimmed messages list.
    """
    # TODO:
    # YOUR CODE HERE
    return messages


def run_with_memory_managed(model, tokenizer, user_turns):
    """Short-term memory + sliding-window trimming each turn."""
    print("=" * 64)
    print("MEMORY + SLIDING WINDOW")
    print("=" * 64)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in user_turns:
        messages.append({"role": "user", "content": turn})
        # Keep within budget BEFORE generating:
        messages = trim_with_sliding_window(tokenizer, messages)
        reply = generate(model, tokenizer, messages)
        messages.append({"role": "assistant", "content": reply})
        print(f"\n👨‍🦲 User: {turn}")
        print(f"🤖 Bot : {reply}")
        print(f"   [tokens: {count_tokens(tokenizer, messages)} | messages: {len(messages)}]")
    print()
    return messages



**Test**: Now, run the model with the sliding window. The conversation now stays under budget.

What happens when you ask about something from the start of the conversation?

In [ ]:
# run_with_memory(model, tokenizer, CONVERSATION_LONG)          # watch tokens climb
# run_with_memory_managed(model, tokenizer, CONVERSATION_LONG)  # then bound them

## Task 3: Summarize instead of forget (12 mins)

Dropping old turns loses information. 

A better strategy is to compress them.
Implement `summarize_old_turns`:
* Keep the system message and the most recent `keep_recent` turns as-is
* Take all the older messages and ask the model itself to summarize them into a few sentences (use a one-off `generate` call with a summarization instruction). You can try out different instructions and formats (e.g., ask the model to use short bullet points).
* Replace those older messages with a single `system` (or `assistant`) message containing the summary

Plug it into `run_with_memory_summarized` (triggered when over budget, like Task 2).


In [ ]:
def summarize_old_turns(model, tokenizer, messages, keep_recent=2):
    """
    Compress old turns instead of dropping them.
      - Keep messages[0] (system) and the last `keep_recent` messages untouched.
      - Summarize everything in between by asking the model.
      - Replace those middle messages with ONE summary message.
    Returns the new (shorter) messages list.
    """
    # TODO:
    #   1. Split
    #   2. If there's nothing old to summarize, return messages unchanged.
    #   3. Build a summarization prompt and generate a summary.
    #   4. Return: [system, {"role":"system","content":"Earlier conversation summary:\n"+summary}, *recent]
    
    # YOUR CODE HERE
    
    return messages


def run_with_memory_summarized(model, tokenizer, user_turns):
    """Short-term memory + summarize-when-over-budget."""
    print("=" * 64)
    print("MEMORY + SUMMARIZATION")
    print("=" * 64)
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in user_turns:
        messages.append({"role": "user", "content": turn})
        # If over budget, compress old turns instead of dropping them:
        if count_tokens(tokenizer, messages) > MAX_CONTEXT_TOKENS:
            print("   …over budget → summarizing old turns…")
            messages = summarize_old_turns(model, tokenizer, messages)
        reply = generate(model, tokenizer, messages)
        messages.append({"role": "assistant", "content": reply})
        print(f"\n👨‍🦲 User: {turn}")
        print(f"🤖 Bot : {reply}")
        print(f"   [tokens: {count_tokens(tokenizer, messages)} | messages: {len(messages)}]")
    print()
    return messages


**Test**: run your solution on `CONVERSATION_LONG`. The token count should stay bounded and the model should still recall key early facts (because they survive in the summary), unlike the previous sliding-window version.

In [ ]:
# run_with_memory_summarized(model, tokenizer, CONVERSATION_LONG)

## Task 4: Memory across conversations (10 mins)

Short-term memory disappears when a conversation ends. Let's make some of it **persist**.


Implement `extract_memory`: given a finished conversation, ask the model to extract a short list of durable facts about the user (name, preferences, ongoing plans) — the kind of thing worth remembering next time. You can play around with the level and kind of information to retain.


Then:
1. Run `CONVERSATION_1` with memory: call `extract_memory` on it; you get a "user profile"
2. Start `CONVERSATION_2` and **inject the extracted profile into the system prompt**.
3. Hint: The provided `run_second_conversation` already does the wiring. You just supply `extract_memory`.


In [ ]:
def extract_memory(model, tokenizer, messages):
    """
    Ask the model to extract durable facts about the user from a finished
    conversation. Returns a short generated text ("user profile").
    """
    # TODO:
    # YOUR CODE HERE
    return ""

In [ ]:
def run_second_conversation(model, tokenizer, user_turns, user_profile):
    """A fresh conversation that starts WITH the carried-over user profile injected."""
    print("=" * 64)
    print("CONVERSATION 2 (fresh) — with injected memory")
    print("=" * 64)
    system = SYSTEM_PROMPT
    if user_profile.strip():
        system += "\n\nWhat you remember about this user:\n" + user_profile
    messages = [{"role": "system", "content": system}]
    for turn in user_turns:
        messages.append({"role": "user", "content": turn})
        reply = generate(model, tokenizer, messages)
        messages.append({"role": "assistant", "content": reply})
        print(f"\n👨‍🦲 User: {turn}")
        print(f"🤖 Bot : {reply}")
    print()
    return messages

**Test:** `CONVERSATION_2` asks a question that only makes sense if the agent remembers the first conversation. What happens with and without the injected memory?

In [ ]:
# convo1 = run_with_memory(model, tokenizer, CONVERSATION_1)
# profile = extract_memory(model, tokenizer, convo1)
# print("---- extracted user profile ----")
# print(profile)
# print("--------------------------------")
# run_second_conversation(model, tokenizer, CONVERSATION_2, profile)

## Bonus Challenges (optional)

1. **Token budget for the summary:** make `summarize_old_turns` summarize only enough
old turns to get back under budget, rather than all of them. Keep as much raw recent context as possible.
2. **Selective memory:** modify `extract_memory` to store facts as structured JSON
(e.g. `{"name": "...", "preferences": [...], "plans": "..."}`) instead of free text. What are the trade-offs vs. a text summary?
3. **Interactive mode:** replace the scripted `user_turns` with a `while True: input()`
loop so you can chat live and watch the token counter in real time.

# Exercise 2: Connecting with a search engine via MCP

In this exercise, you will use the MCP protocol to connect to a search engine Tavily. 

Tavily provides search functionality that LLMs can consume via MCP. First, create an account at [Tavily](https://tavily.com/). It will take no more than a minute.

Tavily provides a free tier where you can perform 1000 searches for free each month. Once you create the account, you will see the API key already created for you. Copy the API key. You will need it later in the exercise.


In [ ]:
!pip install mcp==1.27.2 --quiet

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
import json
import re
from collections import defaultdict
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
TAVILY_API_KEY = "" # Add your key here

In [ ]:
model_name = "Qwen/Qwen3-4B"
device = torch.device("cuda:0")  # Or select a different device
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def convert_to_chat(prompt: str, tokenizer: AutoTokenizer, sys_prompt: str):
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": prompt}
        ],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

### Some helper functions

These functions help you parse the MCP responses. No need to edit these.

They key thing to note is that you can call the `mcp_rpc` function with any parameters, e.g., `tools/list` or `tools/call`. This function will return the server respose to you.


In [ ]:
import itertools
import requests

_session = requests.Session()
_state = {"session_id": None, "protocol_version": None}
_ids = itertools.count(1)


def _parse(resp):
    """Return the JSON-RPC message from a JSON or SSE response (or None)."""
    sid = resp.headers.get("Mcp-Session-Id")
    if sid:
        _state["session_id"] = sid
    ctype = resp.headers.get("Content-Type", "")
    if "text/event-stream" in ctype:
        payload = None
        for line in resp.text.splitlines():
            line = line.strip()
            if line.startswith("data:"):
                data = line[len("data:"):].strip()
                if data:
                    try:
                        payload = json.loads(data)
                    except json.JSONDecodeError:
                        pass
        return payload
    text = resp.text.strip()
    return json.loads(text) if text else None


def mcp_rpc(method, params=None, *, notification=False):
    """Send one JSON-RPC call to the MCP server and return its result message."""
    headers = {
        "Content-Type": "application/json",
        # MCP requires the client to accept BOTH content types:
        "Accept": "application/json, text/event-stream",
    }
    if _state["session_id"]:
        headers["Mcp-Session-Id"] = _state["session_id"]
    if _state["protocol_version"]:
        headers["MCP-Protocol-Version"] = _state["protocol_version"]

    body = {"jsonrpc": "2.0", "method": method}
    if params is not None:
        body["params"] = params
    if not notification:
        body["id"] = next(_ids)

    resp = _session.post(MCP_URL, headers=headers, data=json.dumps(body), timeout=60)
    resp.raise_for_status()

    msg = _parse(resp)
    if msg and "error" in msg:
        raise RuntimeError(f"MCP error for {method}: {msg['error']}")
    return msg

### Connecting to the Tavily MCP server

Taily provides you with a [URL you can use to connect to the MCP server](https://github.com/tavily-ai/tavily-mcp?tab=readme-ov-file#remote-mcp-server). Every MCP server will have its own URL.

In [ ]:
MCP_URL = f"https://mcp.tavily.com/mcp/?tavilyApiKey={TAVILY_API_KEY}"

## Connecting to the MCP Server

Opening an MCP connection involves two steps:

1. Send an `initialize` request
2. send the `notifications/initialized` notification to say the handshake is done.

In [ ]:
init = mcp_rpc("initialize", {
    "protocolVersion": "2025-06-18",
    "capabilities": {},
    "clientInfo": {"name": "my-first-mcp-client", "version": "1.0.0"},  # Feel free to add the name of your own liking
})

# Remember the protocol version the server negotiated, then finish the handshake.
_state["protocol_version"] = init["result"].get("protocolVersion")
mcp_rpc("notifications/initialized", notification=True)

server = init["result"].get("serverInfo", {})
print("Connected to:", server.get("name"), server.get("version"))
print("Protocol version:", _state["protocol_version"])
print("Session id:", _state["session_id"])

## Listing the available tools (`tools/list`)

Let us ask the server to list the tools it offers. We will issue a `tools/list` request and print the tools it offers. See the [MCP protocol documentation](https://modelcontextprotocol.io/docs/learn/server-concepts#how-tools-work) for more details.


In [ ]:
import json
tools = mcp_rpc("tools/list")["result"]["tools"]
print(json.dumps(tools, indent=4))

Let us parse this json to get the tool names and descriptions.

See the example in our lecture slides or at the [MCP protocol documentation](https://modelcontextprotocol.io/docs/learn/server-concepts#how-tools-work) to learn how the json is structured.

Most imporrantly, evey tool exposes a `name` a `description` and an `inputSchema`.

In [ ]:
print("== Tools offered by the Tavily MCP server ==")

for i, t in enumerate(tools):
    name = t.get("name")
    desc = (t.get("description") or "").split(".")[0]
    print(f"{i+1} - {t['name']}: {desc}")
    params = t.get("inputSchema", {}).get("properties", {})
    required = t.get("inputSchema", {}).get("required", [])
    print(f" * Required parameters: {required} *")
    for p in params:
        print(f"    * {p}: {params[p].get('description')}")

## Calling a specific tool (`tools/call`)

Let us now call a specific tool. We need to use the name of the tool and pass the required parameters.

In [ ]:
query = "What is the capital of Germany?"

server_input = {
    "name": "tavily_search",  # This is the name of the tool we want to call
    "arguments": {
        "query": query,      # take the prompt straight from the user
        "max_results": 5,    # optional parameter
    },
}

output = mcp_rpc("tools/call", server_input)

In [ ]:
# A tools/call result is a list of content blocks; keep the text ones.
blocks = output["result"]["content"]
search_text = "\n".join(b["text"] for b in blocks if b.get("type") == "text")

print(json.dumps(output, indent=4))


## Equipping Qwen with Tavily Search

Your task is to connect the Qwen model with the Tavily search engine. Given a user query, write the code to connect to the search engine and retrieve the results.

In [ ]:
# Your code here